# Domain keyword clustering (UMAP)

Report Section 4.3.1, Step 1.2 and Figures 5.2-5.3.

1. Each domain (Government, Health, IoT) gets a **centroid**: the embedding of its textual description.
2. All 300 domain keywords and the 30 general keywords are embedded with `all-MiniLM-L6-v2`
   (sentence-transformers, runs locally - **no API key needed**).
3. UMAP projects everything to 2-D (Figure 5.2).
4. Cosine similarity to the domain centroid selects the **top 30 keywords per domain** (Table 5.2, Figure 5.3).

**Inputs** (`keyword_inputs/`): `domain.json` (domain descriptions), `domain_keywords.csv` (100 candidate keywords per domain),
`general_keywords.csv` (30 general privacy keywords).

**Outputs**: `figures/keyword_clustering_umap.png`, `figures/top30_keywords_umap.png`, `top30_keywords_per_domain.csv`.

Run from this folder (`01_keywords/`), after `pip install sentence-transformers umap-learn scikit-learn matplotlib pandas`.
The figures in `figures/` are from the original run; UMAP layouts can differ slightly between library versions.

In [ ]:
import json
import os

import pandas as pd

## 1. Load the domains and keywords

In [ ]:
DOMAIN_JSON = "keyword_inputs/domain.json"
SPECIFIC_CSV = "keyword_inputs/domain_keywords.csv"
GENERAL_CSV = "keyword_inputs/general_keywords.csv"

with open(DOMAIN_JSON, "r", encoding="utf-8") as f:
    domain_data = json.load(f)

domains = [d["Domain"] for d in domain_data]
print(domains)

df_specific = pd.read_csv(SPECIFIC_CSV)
df_general = pd.read_csv(GENERAL_CSV)

df_specific["Keyword"] = df_specific["Keyword"].astype(str).str.strip()
df_specific["Domain"]  = df_specific["Domain"].astype(str).str.strip()
df_general["Keyword"]  = df_general["Keyword"].astype(str).str.strip()

domains = [d["Domain"] for d in domain_data]
domain_desc = {d["Domain"]: d.get("Description", "") for d in domain_data}

print("Domains:", domains)
print(df_specific.head())
print(df_general.head())

num_rows_specific = len(df_specific)
print(f"The DataFrame 'df_specific' has {num_rows_specific} rows.")

## 2. Domain centroid texts
Each domain is represented by its description only (the keywords are not part of the centroid).

In [ ]:
domain_texts = {}

for d in domain_data:
    dom = d["Domain"]
    desc = d.get("Description", "").strip()

    if not desc:
        raise ValueError(f"Domain {dom} has no description")

    domain_texts[dom] = [desc]

In [ ]:
# Check the domain descriptions
for dom, texts in domain_texts.items():
    print(dom, texts)

## 3. Embed centroids and keywords

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# centroids
domain_centroids = {}
for dom, texts in domain_texts.items():
    emb = model.encode(texts, normalize_embeddings=True)
    domain_centroids[dom] = emb.mean(axis=0)

centroid_matrix = np.vstack([domain_centroids[d] for d in domains])

# specific points
specific_vectors = model.encode(df_specific["Keyword"].tolist(), normalize_embeddings=True)

# general points
general_vectors = model.encode(df_general["Keyword"].tolist(), normalize_embeddings=True)

## 4. UMAP projection to 2-D

In [ ]:
import umap
import numpy as np

all_vectors = np.vstack([centroid_matrix, specific_vectors, general_vectors])

reducer = umap.UMAP(
    n_neighbors=100,
    min_dist=0.001,
    metric="cosine",
    random_state=42
)

all_2d = reducer.fit_transform(all_vectors)

n_centroids = len(domains)
n_specific  = len(df_specific)

centroids_2d = all_2d[:n_centroids]
specific_2d  = all_2d[n_centroids:n_centroids+n_specific]
general_2d   = all_2d[n_centroids+n_specific:]

## 5. Figure 5.2 - domain centroids and keyword clusters

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

for dom in domains:
    mask = (df_specific["Domain"].values == dom)
    plt.scatter(
        specific_2d[mask, 0], specific_2d[mask, 1],
        s=25, alpha=0.7, label=f"Specific: {dom}"
    )

# general points (gray-ish default)
plt.scatter(general_2d[:, 0], general_2d[:, 1], s=25, alpha=0.35, label="General")

# centroids (big X)
for i, dom in enumerate(domains):
    x, y = centroids_2d[i]
    plt.scatter([x], [y], marker="X", s=260, edgecolor="black", linewidth=1.2, zorder=10)
    plt.text(x, y, f"  {dom}", fontsize=12, weight="bold")

plt.title("Domain centroids and keyword clustering analysis")
plt.legend()
plt.grid(True, alpha=0.2)
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/keyword_clustering_umap.png", bbox_inches="tight")
plt.show()

## 6. Top 30 keywords per domain (cosine similarity to the centroid)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def top_k_keywords_per_domain_filtered(df_specific, specific_vectors, domains, centroid_matrix, k=30):
    results = []

    for i, dom in enumerate(domains):
        # Centroid of this domain
        domain_centroid = centroid_matrix[i].reshape(1, -1)

        # Keywords labelled with this domain
        mask = df_specific["Domain"] == dom
        if not mask.any():
            continue

        filtered_keywords = df_specific[mask]["Keyword"].tolist()
        filtered_vectors = specific_vectors[mask]

        # Cosine similarity (dot product, vectors are normalised)
        similarities = np.dot(filtered_vectors, domain_centroid.T).flatten()

        sorted_indices = np.argsort(similarities)[::-1]

        # Keep the top K (or all, if fewer than K)
        top_k_indices = sorted_indices[:min(k, len(filtered_keywords))]

        for idx in top_k_indices:
            results.append({
                "Domain": dom,
                "Keyword": filtered_keywords[idx],
                "Similarity": round(float(similarities[idx]), 4)
            })

    return pd.DataFrame(results)

In [ ]:
df_top_specific = top_k_keywords_per_domain_filtered(
    df_specific=df_specific,
    specific_vectors=specific_vectors,
    domains=domains,
    centroid_matrix=centroid_matrix,
    k=30
)

df_top_specific.to_csv("top30_keywords_per_domain.csv", index=False, encoding='utf-8-sig')

print(df_top_specific.head(30))

## 7. Figure 5.3 - top 30 keywords highlighted

In [ ]:
# All candidate keywords (100 per domain)
all_keywords = df_specific

# Top 30 per domain
top30 = df_top_specific

# Pairs to highlight
top30_set = set(zip(top30["Keyword"], top30["Domain"]))

plt.figure()

for dom in domains:
    mask = (df_specific["Domain"] == dom)

    # --- ALL 100 (background) ---
    plt.scatter(
        specific_2d[mask, 0],
        specific_2d[mask, 1],
        s=20, alpha=0.2, label=f"{dom} (all)"
    )

    # --- TOP 30 (highlight) ---
    highlight_mask = [
        (kw, d) in top30_set
        for kw, d in zip(df_specific["Keyword"], df_specific["Domain"])
    ]
    highlight_mask = (df_specific["Domain"] == dom) & highlight_mask

    plt.scatter(
        specific_2d[highlight_mask, 0],
        specific_2d[highlight_mask, 1],
        s=60, alpha=0.9, edgecolor='black', label=f"{dom} (top30)"
    )

# centroid
for i, dom in enumerate(domains):
    x, y = centroids_2d[i]
    plt.scatter([x], [y], marker="X", s=250, edgecolor="black")
    plt.text(x, y, dom, fontsize=12, weight="bold")

plt.title("Top 30 Keywords by Cosine Similarity to Domain Centroid")
plt.legend()
plt.savefig("figures/top30_keywords_umap.png", bbox_inches="tight")
plt.show()

## 8. Mean similarity of each domain's keywords to its centroid

In [ ]:
for dom in domains:
    mask = df_specific["Domain"] == dom
    vecs = specific_vectors[mask]
    centroid = domain_centroids[dom]

    sim = np.dot(vecs, centroid)  # cosine similarity (normalised vectors)
    print(f"{dom}: mean_similarity = {sim.mean():.4f}")